# Duvar — Point-Based Neural Mapping Framework

> *"Like a wall built from bricks — each brick independent, each brick replaceable, each brick aware of where the next one sits."*

**License:** GNU General Public License v3.0  
**Architecture:** PBNM-Flow (Point-Based Neural Mapping)  
**Reference model:** TinyLlama-1.1B-Chat-v1.0  
**Runtime:** Google Colab T4 GPU (16 GB VRAM)

---

## What Is Duvar?

A standard transformer forward pass is a fixed loop: `for layer in self.layers`. The model does not decide which blocks to visit — Python does, by iterating a list in order.

Duvar breaks this. Each transformer block is assigned a `target_ptr` — an explicit integer pointer to its successor. A custom forward function reads those pointers at runtime and traverses the graph accordingly. The execution path becomes a first-class artifact: a concrete, logged, reconfigurable sequence of block indices.

This notebook demonstrates the routing mechanism on TinyLlama-1.1B. No quantization. No kernel tricks. Only the traversal logic.

## Architecture

| Component | Role |
|---|---|
| `DuvarBlockGraph` | Directed graph over transformer decoder blocks. Maps each block index to its successor. |
| `duvar_forward()` | Custom forward pass. Traverses blocks by reading `DuvarBlockGraph` pointers instead of iterating a fixed list. |
| `duvar_generate()` | Custom generation loop. Calls `duvar_forward` at every token step and logs the execution path. |

## Routing Modes

**Sequential:** `0 → 1 → 2 → ... → 21 → OUTPUT`  
Identical to standard execution. Used as a parity check.

**Sparse:** `set_route({9: 16})` — jump from block 9 to block 16.  
Blocks 10–15 are bypassed. Route log: `[0,1,2,3,4,5,6,7,8,9,16,17,18,19,20,21]`

## Theoretical Claims

1. **Topology over arithmetic**: the execution path is a runtime parameter, not a code constant. Only the blocks declared in the active path are computed.
2. **Sparse execution potential**: bypassing blocks reduces active operation count proportionally. Measurable latency reduction demonstrated in Cell 9.
3. **Intrinsic XAI**: the route log produced by `duvar_generate()` is an exact record of which blocks participated in every token decision — not an approximation reconstructed from gradients.

## Honest Limitations

TinyLlama on a T4 demonstrates the mechanism. The gains from sparse routing become architecturally significant at 70B+ scale and distributed inference. The current `duvar_generate()` does not implement a KV cache — this is deliberate for clarity; pointer routing and KV cache are orthogonal.

## Cell 1 — Environment Setup

Run once. Takes ~60 seconds on a Colab T4.

In [8]:
import subprocess, sys

PACKAGES = [
    "transformers==4.40.0",
    "accelerate",
    "sentencepiece",
    "protobuf",
    "huggingface_hub",
]

for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("Setup complete.")

Setup complete.


## Cell 2 — Imports and GPU Verification

In [9]:
import torch
import torch.nn.functional as F
import numpy as np
import time, os, json
from typing import Optional, List
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

print(f"PyTorch : {torch.__version__}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. In Colab: Runtime -> Change Runtime Type -> T4 GPU"
    )

DEVICE = torch.device("cuda")
gpu    = torch.cuda.get_device_properties(0)

print(f"GPU     : {gpu.name}")
print(f"VRAM    : {gpu.total_memory / 1e9:.1f} GB")
print(f"SMs     : {gpu.multi_processor_count}")
print(f"Compute : {gpu.major}.{gpu.minor}")
print("\nEnvironment ready.")

PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB
SMs     : 40
Compute : 7.5

Environment ready.


## Cell 3 — Download Model

First run: ~2.2 GB download. Subsequent runs use the Colab cache.

In [10]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Downloading: {MODEL_ID}")
model_cache = snapshot_download(
    repo_id=MODEL_ID,
    ignore_patterns=[
        "*.msgpack", "*.h5", "flax_model*",
        "tf_model*", "rust_model*", "*.ot",
    ],
)
print(f"Cached at: {model_cache}")

Downloading: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Cached at: /root/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6


## Cell 4 — Load Model

Standard FP16 load. No conversion, no quantization. The model is used as-is; only the *traversal* changes.

In [11]:
print("Loading TinyLlama-1.1B (FP16)...")

tokenizer = AutoTokenizer.from_pretrained(model_cache)
model     = AutoModelForCausalLM.from_pretrained(
    model_cache,
    torch_dtype=torch.float16,
    device_map="cuda",
    low_cpu_mem_usage=True,
)
model.eval()

cfg   = model.config
n_par = sum(p.numel() for p in model.parameters())

print(f"\nModel info:")
print(f"  Architecture  : {cfg.model_type}")
print(f"  Parameters    : {n_par / 1e9:.3f}B")
print(f"  Hidden dim    : {cfg.hidden_size}")
print(f"  Decoder blocks: {cfg.num_hidden_layers}")
print(f"  KV heads      : {cfg.num_key_value_heads} (GQA)")
print(f"  Dtype         : FP16")

NUM_LAYERS = cfg.num_hidden_layers
print(f"\nDuvar will build a block graph over {NUM_LAYERS} decoder blocks.")

Loading TinyLlama-1.1B (FP16)...

Model info:
  Architecture  : llama
  Parameters    : 1.100B
  Hidden dim    : 2048
  Decoder blocks: 22
  KV heads      : 4 (GQA)
  Dtype         : FP16

Duvar will build a block graph over 22 decoder blocks.


## Cell 5 — DuvarBlockGraph

The routing table. Maps each block index to its successor at runtime.

- Default: sequential `0 → 1 → 2 → ... → N-1 → 'OUTPUT'`
- `set_route(patches)`: override any pointer with a single dict update
- `active_path()`: return the exact sequence of blocks that will execute — before inference begins
- `reset()`: restore full sequential routing

The graph is the only thing that controls execution order. `duvar_forward()` reads it and nothing else.

In [12]:
class DuvarBlockGraph:
    """
    Directed routing graph over transformer decoder blocks.

    Each entry maps a block index (int) to its successor (int or 'OUTPUT').
    The default graph is sequential. Any pointer can be patched at runtime
    without touching model weights.
    """

    def __init__(self, num_layers: int):
        self.num_layers = num_layers
        self._default   = {
            i: (i + 1 if i + 1 < num_layers else "OUTPUT")
            for i in range(num_layers)
        }
        self.routing = dict(self._default)

    # ── Core API ──────────────────────────────────────────────────────────

    def successor(self, idx: int):
        """Return the successor of block `idx`."""
        return self.routing.get(idx, "OUTPUT")

    def set_route(self, patches: dict):
        """
        Patch one or more pointers.

        Example:
            graph.set_route({9: 16})   # skip blocks 10-15
            graph.set_route({4: 8, 12: 18})
        """
        self.routing.update(patches)

    def reset(self):
        """Restore full sequential routing."""
        self.routing = dict(self._default)

    def active_path(self) -> List[int]:
        """
        Return the ordered list of block indices that will be executed
        given the current routing table.
        Terminates at 'OUTPUT'.
        """
        path, current = [], 0
        while current != "OUTPUT":
            path.append(current)
            current = self.successor(current)
        return path

    # ── Serialization ─────────────────────────────────────────────────────

    def to_dict(self) -> dict:
        return {
            "num_layers":   self.num_layers,
            "routing":      {str(k): str(v) for k, v in self.routing.items()},
            "active_path":  self.active_path(),
            "skipped":      sorted(
                set(range(self.num_layers)) - set(self.active_path())
            ),
        }

    def __repr__(self):
        path    = self.active_path()
        skipped = sorted(set(range(self.num_layers)) - set(path))
        return (
            f"DuvarBlockGraph({self.num_layers} blocks | "
            f"active={len(path)} | skipped={skipped or 'none'})"
        )


# ── Smoke test ────────────────────────────────────────────────────────────
g = DuvarBlockGraph(4)
assert g.active_path() == [0, 1, 2, 3]
g.set_route({1: 3})
assert g.active_path() == [0, 1, 3]
g.reset()
assert g.active_path() == [0, 1, 2, 3]
print("DuvarBlockGraph — OK")
print(DuvarBlockGraph(NUM_LAYERS))

DuvarBlockGraph — OK
DuvarBlockGraph(22 blocks | active=22 | skipped=none)


## Cell 6 — PBNM-Flow: duvar_forward and duvar_generate

### duvar_forward

Replaces the implicit `for layer in self.layers` loop. The traversal is:

```
current = 0
while current != 'OUTPUT':
    hidden_states = model.layers[current](hidden_states, ...)
    route_log.append(current)
    current = block_graph.successor(current)
```

The `block_graph` argument is the only thing that determines which blocks run and in what order.

### duvar_generate

Wraps `duvar_forward` into a full autoregressive generation loop. Greedy decoding with temperature and top-p. Returns both the generated text and the route log of the **last token step** — the execution trace of the final decision.

In [13]:
def _causal_mask(seq_len: int, device, dtype) -> torch.Tensor:
    """Upper-triangular causal attention mask (additive, -inf above diagonal)."""
    mask = torch.full((seq_len, seq_len), float("-inf"), device=device, dtype=dtype)
    return torch.triu(mask, diagonal=1).unsqueeze(0).unsqueeze(0)


def duvar_forward(
    model,
    input_ids:   torch.Tensor,
    block_graph: DuvarBlockGraph,
    route_log:   Optional[List[int]] = None,
) -> torch.Tensor:
    """
    Forward pass driven by DuvarBlockGraph pointers.
    """
    device  = input_ids.device
    seq_len = input_ids.shape[1]
    dtype   = model.dtype

    # Embed tokens
    hidden_states = model.model.embed_tokens(input_ids)                  # (1, S, H)

    # Positional ids and causal mask
    position_ids = torch.arange(seq_len, device=device).unsqueeze(0)    # (1, S)
    attn_mask    = _causal_mask(seq_len, device, dtype)                 # (1,1,S,S)

    # ── Pointer-driven traversal ───────────────────────────────────────────
    current = 0
    while current != "OUTPUT":
        layer         = model.model.layers[current]
        # Ensure hidden_states stays in model dtype
        hidden_states = layer(
            hidden_states,
            attention_mask=attn_mask,
            position_ids=position_ids,
        )[0]
        if route_log is not None:
            route_log.append(current)
        current = block_graph.successor(current)
    # ──────────────────────────────────────────────────────────────────────

    hidden_states = model.model.norm(hidden_states)                      # final RMSNorm
    return model.lm_head(hidden_states)                                  # (1, S, V)

# ... (duvar_generate implementation remains the same)
def duvar_generate(
    model,
    tokenizer,
    prompt:      str,
    block_graph: DuvarBlockGraph,
    max_new_tokens: int  = 200,
    temperature:    float = 0.7,
    top_p:          float = 0.9,
    seed:           int   = 42,
) -> dict:
    torch.manual_seed(seed)
    chat   = [{"role": "user", "content": prompt}]
    prompt_str = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    input_ids = tokenizer(prompt_str, return_tensors="pt").input_ids.to(DEVICE)
    generated = []
    last_route: List[int] = []
    t0 = time.time()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            last_route = []
            logits     = duvar_forward(model, input_ids, block_graph, route_log=last_route)
            next_logits = logits[0, -1, :] / max(temperature, 1e-5)
            probs  = torch.softmax(next_logits.float(), dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cumsum = torch.cumsum(sorted_probs, dim=0)
            cutoff = (cumsum - sorted_probs) > top_p
            sorted_probs[cutoff] = 0.0
            sorted_probs /= sorted_probs.sum()
            next_token = sorted_idx[torch.multinomial(sorted_probs, 1)].unsqueeze(0)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=1)
    elapsed = time.time() - t0
    return {
        "text":      tokenizer.decode(generated, skip_special_tokens=True),
        "route_log": last_route,
        "n_tokens":  len(generated),
        "elapsed_s": elapsed,
    }

print("duvar_forward + duvar_generate — fixed and reloaded.")

duvar_forward + duvar_generate — fixed and reloaded.


## Cell 7 — Sequential Routing (Parity Check)

Default graph: `0 → 1 → 2 → ... → 21 → OUTPUT`. Every block runs in order. This should produce output indistinguishable from `model.generate()` given the same seed.

In [14]:
TEST_PROMPT = (
    "Explain the architectural differences between a transformer "
    "decoder and a recurrent neural network. Be concise."
)

seq_graph = DuvarBlockGraph(NUM_LAYERS)

print(seq_graph)
print(f"Active path : {seq_graph.active_path()}")
print(f"Skipped     : none")
print()

seq_result = duvar_generate(
    model, tokenizer, TEST_PROMPT,
    block_graph=seq_graph,
    max_new_tokens=200,
)

print("=" * 60)
print("SEQUENTIAL ROUTING OUTPUT")
print("=" * 60)
print(seq_result["text"])
print()
print(f"Tokens     : {seq_result['n_tokens']}")
print(f"Time       : {seq_result['elapsed_s']:.1f}s")
print(f"Route (last token): {seq_result['route_log']}")

DuvarBlockGraph(22 blocks | active=22 | skipped=none)
Active path : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Skipped     : none

SEQUENTIAL ROUTING OUTPUT
Transformer decoders and recurrent neural networks (RNNs) differ in terms of their architectures. Here's a brief explanation of the main differences:

1. Transformers: Transformers are a family of transformer architectures that use a self-attention mechanism to process sequences. They are designed to model the way humans process language, with attention weights that attend to specific parts of the input sequence. Transformers are commonly used for language translation, speech recognition, and natural language processing.

2. Recurrent Neural Networks: RNNs are a type of neural network that are designed to process sequential data. They are commonly used for tasks such as speech recognition, text classification, and time series prediction. RNNs are characterized by their ability to store and retrie

## Cell 8 — Sparse Routing (Skip Demo)

Patch one pointer: `set_route({9: 16})`. Blocks 10 through 15 are skipped entirely. No weight changes. No retraining. The only modification is a single dictionary entry in `DuvarBlockGraph.routing`.

The route log confirms which blocks actually ran.

In [15]:
sparse_graph = DuvarBlockGraph(NUM_LAYERS)
sparse_graph.set_route({9: 16})   # skip blocks 10, 11, 12, 13, 14, 15

print(sparse_graph)
print(f"Active path : {sparse_graph.active_path()}")
print(f"Skipped     : {sorted(set(range(NUM_LAYERS)) - set(sparse_graph.active_path()))}")
print()

sparse_result = duvar_generate(
    model, tokenizer, TEST_PROMPT,
    block_graph=sparse_graph,
    max_new_tokens=200,
)

print("=" * 60)
print("SPARSE ROUTING OUTPUT  (blocks 10-15 skipped)")
print("=" * 60)
print(sparse_result["text"])
print()
print(f"Tokens     : {sparse_result['n_tokens']}")
print(f"Time       : {sparse_result['elapsed_s']:.1f}s")
print(f"Route (last token): {sparse_result['route_log']}")

DuvarBlockGraph(22 blocks | active=16 | skipped=[10, 11, 12, 13, 14, 15])
Active path : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 16, 17, 18, 19, 20, 21]
Skipped     : [10, 11, 12, 13, 14, 15]

SPARSE ROUTING OUTPUT  (blocks 10-15 skipped)
The model contains a single fully-connected layer. The model should provide the necessary information for the classification task, including the results of the model. Apart from that, the model should be able to use this information to generate speech representations for the text. The model should be able to generate the speech that is used for speaking the language. The model should generate the text from the language spoken by a person who uses a speech synthesis system that is controlled by a language model. The model should be able to generate the text that is controlled by the model. The model should be able to generate the text that is controlled by the model. 

The model should be able to generate the text that is controlled by the model. Additionally, t

## Cell 9 — System Report

Side-by-side comparison: sequential vs sparse routing.

In [16]:
# ── Output similarity ─────────────────────────────────────────────────────
seq_words    = set(seq_result["text"].lower().split())
sparse_words = set(sparse_result["text"].lower().split())
jaccard      = len(seq_words & sparse_words) / max(len(seq_words | sparse_words), 1)

seq_path    = seq_graph.active_path()
sparse_path = sparse_graph.active_path()
skipped     = sorted(set(range(NUM_LAYERS)) - set(sparse_path))

speedup = seq_result["elapsed_s"] / max(sparse_result["elapsed_s"], 1e-6)

print("=" * 62)
print("DUVAR SYSTEM REPORT — TinyLlama-1.1B")
print("=" * 62)
print(f"""
ROUTING
  Sequential path   : {seq_path}
  Sparse path       : {sparse_path}
  Skipped blocks    : {skipped}
  Blocks active (seq)   : {len(seq_path)} / {NUM_LAYERS}
  Blocks active (sparse): {len(sparse_path)} / {NUM_LAYERS}

SPEED
  Sequential        : {seq_result['elapsed_s']:.1f}s  ({seq_result['n_tokens']} tokens)
  Sparse            : {sparse_result['elapsed_s']:.1f}s  ({sparse_result['n_tokens']} tokens)
  Speedup           : {speedup:.2f}x

OUTPUT QUALITY
  Sequential words  : {len(seq_result['text'].split())}
  Sparse words      : {len(sparse_result['text'].split())}
  Jaccard similarity: {jaccard * 100:.1f}%
  Note: Jaccard is a word-overlap proxy.
        Semantic drift is expected when 6 of 22 blocks are bypassed.

ARCHITECTURE
  Model             : TinyLlama-1.1B-Chat-v1.0  (FP16, unmodified)
  Routing mechanism : DuvarBlockGraph (pointer-driven traversal)
  Forward pass      : duvar_forward  (custom while-loop over block_graph)
  Generation loop   : duvar_generate (no KV cache — clarity trade-off)
""")

print("ROUTE LOG (last token step)")
print("-" * 62)
print(f"  Sequential : {seq_result['route_log']}")
print(f"  Sparse     : {sparse_result['route_log']}")
print("=" * 62)

DUVAR SYSTEM REPORT — TinyLlama-1.1B

ROUTING
  Sequential path   : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
  Sparse path       : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 16, 17, 18, 19, 20, 21]
  Skipped blocks    : [10, 11, 12, 13, 14, 15]
  Blocks active (seq)   : 22 / 22
  Blocks active (sparse): 16 / 22

SPEED
  Sequential        : 12.9s  (200 tokens)
  Sparse            : 4.9s  (200 tokens)
  Speedup           : 2.63x

OUTPUT QUALITY
  Sequential words  : 139
  Sparse words      : 176
  Jaccard similarity: 11.9%
  Note: Jaccard is a word-overlap proxy.
        Semantic drift is expected when 6 of 22 blocks are bypassed.

ARCHITECTURE
  Model             : TinyLlama-1.1B-Chat-v1.0  (FP16, unmodified)
  Routing mechanism : DuvarBlockGraph (pointer-driven traversal)
  Forward pass      : duvar_forward  (custom while-loop over block_graph)
  Generation loop   : duvar_generate (no KV cache — clarity trade-off)

ROUTE LOG (last token step)
-----------------

## Cell 10 — Latency Benchmark

Per-token latency using `torch.cuda.Event` for sub-millisecond precision. 10 iterations each. The warmup step is excluded from the average.

Expected: sparse routing is faster in proportion to the fraction of blocks bypassed.

In [17]:
BENCH_PROMPT = (
    "How does pointer-based graph traversal differ from standard "
    "sequential neural network execution?"
)

def benchmark_graph(
    graph:      DuvarBlockGraph,
    label:      str,
    iterations: int = 10,
    tokens_per_iter: int = 32,
) -> float:
    """
    Measure mean per-token latency (ms) over `iterations` runs.
    Uses torch.cuda.Event for GPU-side timing.
    """
    chat      = [{"role": "user", "content": BENCH_PROMPT}]
    prompt_s  = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    input_ids = tokenizer(prompt_s, return_tensors="pt").input_ids.to(DEVICE)

    # Warmup
    with torch.no_grad():
        _ = duvar_forward(model, input_ids, graph)
    torch.cuda.synchronize()

    latencies = []
    for i in range(iterations):
        # Generate `tokens_per_iter` tokens, time the whole run
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()

        ids = input_ids.clone()
        with torch.no_grad():
            for _ in range(tokens_per_iter):
                logits     = duvar_forward(model, ids, graph)
                next_token = logits[0, -1, :].argmax(dim=-1, keepdim=True).unsqueeze(0)
                ids        = torch.cat([ids, next_token], dim=1)

        end.record()
        torch.cuda.synchronize()

        ms_per_tok = start.elapsed_time(end) / tokens_per_iter
        latencies.append(ms_per_tok)
        print(f"  [{label}] iter {i+1:2d}: {ms_per_tok:.2f} ms/token")

    mean_ms = float(np.mean(latencies))
    print(f"  → mean: {mean_ms:.2f} ms/token\n")
    return mean_ms


print("Benchmarking sequential routing...")
seq_ms = benchmark_graph(seq_graph, label="seq")

print("Benchmarking sparse routing (blocks 10-15 skipped)...")
sp_ms = benchmark_graph(sparse_graph, label="sparse")

print("=" * 50)
print(f"  Sequential  : {seq_ms:.2f} ms/token")
print(f"  Sparse      : {sp_ms:.2f} ms/token")
print(f"  Speedup     : {seq_ms / max(sp_ms, 1e-6):.2f}x")
print(f"  Blocks saved: {NUM_LAYERS - len(sparse_graph.active_path())} of {NUM_LAYERS}")
print("=" * 50)

Benchmarking sequential routing...
  [seq] iter  1: 32.34 ms/token
  [seq] iter  2: 40.97 ms/token
  [seq] iter  3: 44.16 ms/token
  [seq] iter  4: 32.65 ms/token
  [seq] iter  5: 30.26 ms/token
  [seq] iter  6: 31.10 ms/token
  [seq] iter  7: 30.01 ms/token
  [seq] iter  8: 30.16 ms/token
  [seq] iter  9: 30.76 ms/token
  [seq] iter 10: 30.50 ms/token
  → mean: 33.29 ms/token

Benchmarking sparse routing (blocks 10-15 skipped)...
  [sparse] iter  1: 22.38 ms/token
  [sparse] iter  2: 22.66 ms/token
  [sparse] iter  3: 22.25 ms/token
  [sparse] iter  4: 24.93 ms/token
  [sparse] iter  5: 29.89 ms/token
  [sparse] iter  6: 31.65 ms/token
  [sparse] iter  7: 24.07 ms/token
  [sparse] iter  8: 22.19 ms/token
  [sparse] iter  9: 22.80 ms/token
  [sparse] iter 10: 22.04 ms/token
  → mean: 24.49 ms/token

  Sequential  : 33.29 ms/token
  Sparse      : 24.49 ms/token
  Speedup     : 1.36x
  Blocks saved: 6 of 22


## Cell 11 — Save Route Metadata

Serialize the block graphs to `duvar_metadata.json`. The file records the full routing table, active paths, and skipped blocks for both the sequential and sparse configurations — making the execution topology an artifact alongside the model weights.

In [18]:
SAVE_DIR = "/content/duvar_output"
os.makedirs(SAVE_DIR, exist_ok=True)

meta = {
    "format":   "Duvar-v2",
    "license":  "GPL-3.0",
    "model_id": MODEL_ID,
    "graphs": {
        "sequential": seq_graph.to_dict(),
        "sparse":     sparse_graph.to_dict(),
    },
    "benchmark": {
        "sequential_ms_per_token": round(seq_ms, 3),
        "sparse_ms_per_token":     round(sp_ms,  3),
        "speedup":                 round(seq_ms / max(sp_ms, 1e-6), 3),
    },
}

meta_path = os.path.join(SAVE_DIR, "duvar_metadata.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Metadata saved to {meta_path}")
print(json.dumps(meta, indent=2))

Metadata saved to /content/duvar_output/duvar_metadata.json
{
  "format": "Duvar-v2",
  "license": "GPL-3.0",
  "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "graphs": {
    "sequential": {
      "num_layers": 22,
      "routing": {
        "0": "1",
        "1": "2",
        "2": "3",
        "3": "4",
        "4": "5",
        "5": "6",
        "6": "7",
        "7": "8",
        "8": "9",
        "9": "10",
        "10": "11",
        "11": "12",
        "12": "13",
        "13": "14",
        "14": "15",
        "15": "16",
        "16": "17",
        "17": "18",
        "18": "19",
        "19": "20",
        "20": "21",
        "21": "OUTPUT"
      },
      "active_path": [
        0,
        1,
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9,
        10,
        11,
        12,
        13,
        14,
        15,
        16,
        17,
        18,
        19,
        20,
        21
      ],
      "skipped": []
    },
    "sparse": {
